# Medical Ontology Sample with HPO
## Soft ontology layer, OWA/CWA boundary, three-valued verdicts, and GNN preprocessing

この notebook は **公開の Human Phenotype Ontology (HPO)** を使って、
医学系の小さな knowledge graph / compliance-like validation / GNN 前処理を一通り体験するためのサンプルです。

## この notebook の設計意図

この notebook は、次の示唆を意識して構成しています。

1. **OWA と CWA を明示的に分ける**
   - ontology / graph 側は open-world 的に扱う
   - validation / case completeness は closed-world 的に扱う

2. **soft ontology layer を先に持つ**
   - LLM / extraction / weak supervision 由来の phenotype 候補は、
     いきなり strict OWL assertion にせず、
     まず provenance と confidence を持つ表として管理する

3. **three-valued verdict**
   - `entailed`
   - `contradicted`
   - `not_determined`
   を分ける

4. **vacuous truth を避ける**
   - 「全部 respiratory phenotype である」だけでは empty case が通ってしまう
   - existential requirement を別途置く

5. **最後は GNN 前処理へつなぐ**
   - patient ノード
   - phenotype ノード
   - ontology subclass edge
   - patient-hasPhenotype edge
   を edge table / node table に落とす

## 公開オントロジー
この notebook では HPO の公式 OWL PURL を使います。

- official HPO OWL: `http://purl.obolibrary.org/obo/hp.owl`


## 0. 必要ライブラリ

- `rdflib`
- `pandas`
- `pyshacl`

必要なら次のセルを実行してください。

In [1]:
# 必要なら実行
%pip install pyshacl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/109.8 kB 6.0 MB/s eta 0:00:00


In [2]:
from rdflib import Graph, Namespace, RDF, RDFS, URIRef, Literal
from rdflib.namespace import SH, XSD
import pandas as pd

try:
    from pyshacl import validate
except ImportError as e:
    raise ImportError("pyshacl が見つかりません。必要なら `%pip install pyshacl` を実行してください。") from e

EX = Namespace("http://example.org/")
HP = Namespace("http://purl.obolibrary.org/obo/HP_")

HPO_OWL_URL = "http://purl.obolibrary.org/obo/hp.owl"
print("HPO URL:", HPO_OWL_URL)

HPO URL: http://purl.obolibrary.org/obo/hp.owl


## 1. HPO を読み込む

HPO 全体は大きいので、最初は URL から直接読む想定にしています。
環境によっては少し時間がかかります。

もし読み込みが重い場合は、
- 最初に一度ローカルへ保存
- あるいは必要部分だけをキャッシュ
してください。


In [3]:
g_hpo = Graph()
g_hpo.parse(HPO_OWL_URL)
print("HPO triples:", len(g_hpo))

PluginException: No plugin registered for (application/octet-stream, <class 'rdflib.parser.Parser'>)

## 2. 今回使う phenotype を選ぶ

今回は呼吸器系の簡単な症例例として、次を使います。

- Fever
- Dyspnea
- Cough
- Hypoxemia
- Chest pain

実際の HPO ID は公開 ontology 側から取ります。


In [ ]:
selected_terms = {
    "Fever": "HP_0001945",
    "Dyspnea": "HP_0002094",
    "Cough": "HP_0012735",
    "Hypoxemia": "HP_0012418",
    "ChestPain": "HP_0100749",
}

selected_uris = {k: URIRef(f"http://purl.obolibrary.org/obo/{v}") for k, v in selected_terms.items()}
selected_uris

In [ ]:
def get_label(graph: Graph, uri: URIRef):
    for lbl in graph.objects(uri, RDFS.label):
        return str(lbl)
    return None

term_rows = []
for name, uri in selected_uris.items():
    term_rows.append({
        "short_name": name,
        "uri": str(uri),
        "label": get_label(g_hpo, uri),
    })

terms_df = pd.DataFrame(term_rows)
display(terms_df)

## 3. HPO の subclass edge を一部抽出する

全部を使うと大きいので、ここでは selected term 周辺の `rdfs:subClassOf` だけを見ます。

この edge は後で GNN 前処理にも使います。


In [ ]:
subclass_rows = []
for _, row in terms_df.iterrows():
    child = URIRef(row["uri"])
    for parent in g_hpo.objects(child, RDFS.subClassOf):
        if isinstance(parent, URIRef):
            subclass_rows.append({
                "child": str(child),
                "child_label": get_label(g_hpo, child),
                "parent": str(parent),
                "parent_label": get_label(g_hpo, parent),
            })

subclass_df = pd.DataFrame(subclass_rows)
display(subclass_df.head(50))

## 4. Soft ontology layer を作る

ここでは phenotype extraction の結果を、
**いきなり strict RDF assertion にせず**
まず表で持ちます。

列:
- patient_id
- phenotype_uri
- confidence
- polarity (`present` / `absent`)
- provenance

これは、記事中の
- soft ontology layer
- provenance
- confidence
- periodic reconciliation
の考え方に対応します。


In [ ]:
soft_mentions = pd.DataFrame([
    # P001: severe respiratory-like presentation
    {"patient_id": "P001", "phenotype_uri": str(selected_uris["Fever"]), "confidence": 0.95, "polarity": "present", "provenance": "note_001"},
    {"patient_id": "P001", "phenotype_uri": str(selected_uris["Dyspnea"]), "confidence": 0.91, "polarity": "present", "provenance": "note_001"},
    {"patient_id": "P001", "phenotype_uri": str(selected_uris["Hypoxemia"]), "confidence": 0.88, "polarity": "present", "provenance": "lab_001"},

    # P002: cough only, no fever, no dyspnea
    {"patient_id": "P002", "phenotype_uri": str(selected_uris["Cough"]), "confidence": 0.93, "polarity": "present", "provenance": "note_002"},
    {"patient_id": "P002", "phenotype_uri": str(selected_uris["Fever"]), "confidence": 0.89, "polarity": "absent", "provenance": "note_002"},

    # P003: uncertain dyspnea mention
    {"patient_id": "P003", "phenotype_uri": str(selected_uris["Dyspnea"]), "confidence": 0.52, "polarity": "present", "provenance": "note_003"},
    {"patient_id": "P003", "phenotype_uri": str(selected_uris["ChestPain"]), "confidence": 0.94, "polarity": "present", "provenance": "note_003"},
])

soft_mentions["phenotype_label"] = soft_mentions["phenotype_uri"].map(
    {str(v): get_label(g_hpo, v) for v in selected_uris.values()}
)

display(soft_mentions)

## 5. Soft layer から strict RDF graph に射影する

ここでは簡略化して、
- confidence >= 0.80
- polarity == present
のみを strict assertion として patient graph に射影します。

これは article の
- periodic reconciliation
- selected assertions into a strict fragment
に相当する簡易版です。


In [ ]:
CONF_THRESHOLD = 0.80

strict_mentions = soft_mentions[
    (soft_mentions["confidence"] >= CONF_THRESHOLD) &
    (soft_mentions["polarity"] == "present")
].copy()

display(strict_mentions)

In [ ]:
patient_graph = Graph()
patient_graph.bind("ex", EX)

# classes / properties
patient_graph.add((EX.Patient, RDF.type, RDFS.Class))
patient_graph.add((EX.hasPhenotype, RDF.type, RDF.Property))
patient_graph.add((EX.hasAbsentPhenotype, RDF.type, RDF.Property))

patients = sorted(set(soft_mentions["patient_id"]))

for pid in patients:
    patient_uri = EX[pid]
    patient_graph.add((patient_uri, RDF.type, EX.Patient))

# present assertions
for _, row in strict_mentions.iterrows():
    patient_uri = EX[row["patient_id"]]
    phenotype_uri = URIRef(row["phenotype_uri"])
    patient_graph.add((patient_uri, EX.hasPhenotype, phenotype_uri))

# explicit absent assertions (keep from soft mentions if confidence high enough)
absent_mentions = soft_mentions[
    (soft_mentions["confidence"] >= CONF_THRESHOLD) &
    (soft_mentions["polarity"] == "absent")
].copy()

for _, row in absent_mentions.iterrows():
    patient_uri = EX[row["patient_id"]]
    phenotype_uri = URIRef(row["phenotype_uri"])
    patient_graph.add((patient_uri, EX.hasAbsentPhenotype, phenotype_uri))

print("patient graph triples:", len(patient_graph))

In [ ]:
def graph_to_df(graph: Graph) -> pd.DataFrame:
    rows = [(str(s), str(p), str(o)) for s, p, o in graph]
    return pd.DataFrame(rows, columns=["subject", "predicate", "object"])

display(graph_to_df(patient_graph).sort_values(["subject","predicate","object"]).reset_index(drop=True))

## 6. SHACL で closed-world 的な case completeness を検査する

ここでは SHACL を
**ontology reasoning ではなく data validation**
として使います。

検査内容:
- Patient は phenotype を少なくとも1つ持つこと

これは CWA 的な completeness check です。


In [ ]:
shapes_ttl = """
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix ex: <http://example.org/> .

ex:PatientShape
    a sh:NodeShape ;
    sh:targetClass ex:Patient ;
    sh:property [
        sh:path ex:hasPhenotype ;
        sh:minCount 1 ;
    ] .
"""

shape_graph = Graph()
shape_graph.parse(data=shapes_ttl, format="turtle")
print("shape graph triples:", len(shape_graph))

In [ ]:
conforms, results_graph, results_text = validate(
    data_graph=patient_graph,
    shacl_graph=shape_graph,
    inference="rdfs",
    abort_on_first=False,
    allow_infos=True,
    allow_warnings=True,
)

print("Conforms:", conforms)
print(results_text)

## 7. Three-valued verdict を作る

今回は `SevereRespiratoryPresentation` を、
次の rule-like 条件で定義します。

- Fever present
- Dyspnea present

ただし判定は二値にせず:

- `entailed`: Fever も Dyspnea も strict graph にある
- `contradicted`: Fever または Dyspnea の absent assertion がある
- `not_determined`: どちらも揃わないが、明示的 contradiction もない

これは article にあった
- OWA/CWA の境界
- three-valued feedback
- not entailed != false
の考え方を意識したものです。


In [ ]:
REQ_FEVER = str(selected_uris["Fever"])
REQ_DYSPNEA = str(selected_uris["Dyspnea"])

def has_edge(graph: Graph, s, p, o) -> bool:
    return (s, p, o) in graph

verdict_rows = []
for pid in patients:
    p = EX[pid]
    fever_present = has_edge(patient_graph, p, EX.hasPhenotype, URIRef(REQ_FEVER))
    dysp_present = has_edge(patient_graph, p, EX.hasPhenotype, URIRef(REQ_DYSPNEA))

    fever_absent = has_edge(patient_graph, p, EX.hasAbsentPhenotype, URIRef(REQ_FEVER))
    dysp_absent = has_edge(patient_graph, p, EX.hasAbsentPhenotype, URIRef(REQ_DYSPNEA))

    if fever_present and dysp_present:
        status = "entailed"
        reason = "Both Fever and Dyspnea are present in the strict graph."
    elif fever_absent or dysp_absent:
        status = "contradicted"
        reason = "At least one required phenotype is explicitly marked absent."
    else:
        status = "not_determined"
        reason = "Required phenotypes are not fully entailed, but also not explicitly contradicted."

    verdict_rows.append({
        "patient_id": pid,
        "fever_present": fever_present,
        "dyspnea_present": dysp_present,
        "fever_absent": fever_absent,
        "dyspnea_absent": dysp_absent,
        "status": status,
        "reason": reason,
    })

verdict_df = pd.DataFrame(verdict_rows)
display(verdict_df)

## 8. vacuous truth への注意を確認する

例えば「all observed phenotypes are respiratory-like」という rule だけでは、
phenotype が何も観測されていない patient が通ってしまう可能性があります。

そのため実務では、
- existential requirement (`at least one phenotype`)
- universal-style condition
を分けて考える必要があります。

ここでは SHACL の `minCount 1` が existential 側に相当します。


## 9. HPO を使った patient-phenotype-edge を作る

ここから GNN 前処理に寄せます。

作るもの:
1. patient node table
2. phenotype node table
3. patient -> phenotype edge
4. phenotype -> phenotype subclass edge


In [ ]:
patient_nodes = pd.DataFrame({
    "node_id": [f"patient::{pid}" for pid in patients],
    "node_type": "patient",
    "raw_id": patients,
})

phenotype_nodes = pd.DataFrame({
    "node_id": [f"phenotype::{uri}" for uri in terms_df["uri"]],
    "node_type": "phenotype",
    "raw_id": terms_df["uri"],
    "label": terms_df["label"],
})

display(patient_nodes)
display(phenotype_nodes)

In [ ]:
patient_edges = strict_mentions.copy()
patient_edges["src"] = patient_edges["patient_id"].map(lambda x: f"patient::{x}")
patient_edges["dst"] = patient_edges["phenotype_uri"].map(lambda x: f"phenotype::{x}")
patient_edges["rel"] = "hasPhenotype"
patient_edges = patient_edges[["src", "rel", "dst", "confidence", "provenance"]]
display(patient_edges)

In [ ]:
sub_edges = subclass_df.copy()
sub_edges["src"] = sub_edges["child"].map(lambda x: f"phenotype::{x}")
sub_edges["dst"] = sub_edges["parent"].map(lambda x: f"phenotype::{x}")
sub_edges["rel"] = "subClassOf"
sub_edges = sub_edges[["src", "rel", "dst"]]
display(sub_edges.head(50))

In [ ]:
all_nodes = pd.concat([
    patient_nodes[["node_id", "node_type", "raw_id"]],
    phenotype_nodes[["node_id", "node_type", "raw_id"]],
], ignore_index=True)

all_edges = pd.concat([
    patient_edges[["src", "rel", "dst"]],
    sub_edges[["src", "rel", "dst"]],
], ignore_index=True)

display(all_nodes.head(100))
display(all_edges.head(100))

## 10. integer ID を振って GNN 用 edge table を作る

PyG / DGL / jraph などへ渡す前の典型的な整形です。


In [ ]:
node_index = {nid: i for i, nid in enumerate(all_nodes["node_id"].tolist())}

gnn_edges = all_edges.copy()
gnn_edges["src_id"] = gnn_edges["src"].map(node_index)
gnn_edges["dst_id"] = gnn_edges["dst"].map(node_index)

relation_index = {rel: i for i, rel in enumerate(sorted(gnn_edges["rel"].unique()))}
gnn_edges["rel_id"] = gnn_edges["rel"].map(relation_index)

display(gnn_edges.head(100))
print("relation_index =", relation_index)

## 11. この notebook が前の sample と違う点

前の sample より強い点は次です。

1. **公開の医療オントロジー（HPO）を直接使う**
2. **soft ontology layer** を先に持つ
3. **strict projection** を分ける
4. **OWA/CWA の境界**を明示する
5. **three-valued verdict** を使う
6. **vacuous truth** への注意を構造で反映する
7. **最後を hetero graph / GNN 前処理**で終える

つまり、単なる RDF 読み込みではなく、
**ontology + extraction pipeline + validation + graph learning bridge**
の notebook になっています。


## 12. 次にやるとよい拡張

1. HPO term を増やす
2. Disease Ontology / MONDO と接続する
3. `present / absent / uncertain` を別 edge type にする
4. severity / triage / risk の symbolic rule を足す
5. patient-hpo graph を hetero GNN に入れる
6. explanation 生成を LLM に限定的に担当させる

この方向に伸ばすと、医学系の neuro-symbolic pipeline にかなり近づきます。


## 13. JAX で patient–phenotype link prediction を行う

ここでは、上で作った `patient -> phenotype` の `hasPhenotype` edge を対象に、簡易な link prediction を行います。

目的は次です。

1. 既知の positive edge を train / validation に分割する
2. 存在しない patient–phenotype pair から negative edge をサンプリングする
3. JAX で embedding + logistic decoder を学習する
4. validation set に対する推論スコア、accuracy、AUC、confusion matrix を可視化する

小さな toy graph なので、ここでは GNN message passing そのものよりも、**graph learning の入口としての link prediction 検証セル**に寄せています。

In [ ]:
# 必要な import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import jax
    import jax.numpy as jnp
except ImportError as e:
    raise ImportError(
        "JAX が見つかりません。Colab などでは通常そのまま使えますが、"
        "必要なら `%pip install -U jax jaxlib` を実行してください。"
    ) from e

print("JAX version:", jax.__version__)

In [ ]:
# patient-phenotype の link prediction 用データを作る
# 対象 relation は hasPhenotype のみ。subClassOf は今回は link prediction の対象から外します。

rng = np.random.default_rng(42)

patient_node_ids = all_nodes.loc[all_nodes["node_type"] == "patient", "node_id"].tolist()
phenotype_node_ids = all_nodes.loc[all_nodes["node_type"] == "phenotype", "node_id"].tolist()

positive_pairs = (
    patient_edges[["src", "dst"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

all_candidate_pairs = pd.MultiIndex.from_product(
    [patient_node_ids, phenotype_node_ids],
    names=["src", "dst"]
).to_frame(index=False)

positive_key = set(map(tuple, positive_pairs[["src", "dst"]].to_numpy()))
negative_pairs = all_candidate_pairs[
    ~all_candidate_pairs.apply(lambda r: (r["src"], r["dst"]) in positive_key, axis=1)
].reset_index(drop=True)

# positive edge を train / validation に分割
n_pos = len(positive_pairs)
perm = rng.permutation(n_pos)

# toy data なので validation positive を最低1件、可能なら40%にする
n_val_pos = max(1, int(round(n_pos * 0.4)))
n_val_pos = min(n_val_pos, n_pos - 1)  # train positive を最低1件残す

val_pos = positive_pairs.iloc[perm[:n_val_pos]].copy()
train_pos = positive_pairs.iloc[perm[n_val_pos:]].copy()

# negative は positive と同数程度に揃える
neg_perm = rng.permutation(len(negative_pairs))

n_train_neg = len(train_pos)
n_val_neg = len(val_pos)

train_neg = negative_pairs.iloc[neg_perm[:n_train_neg]].copy()
val_neg = negative_pairs.iloc[neg_perm[n_train_neg:n_train_neg + n_val_neg]].copy()

def make_labeled_edges(pos_df, neg_df):
    pos = pos_df.copy()
    pos["label"] = 1
    neg = neg_df.copy()
    neg["label"] = 0
    out = pd.concat([pos, neg], ignore_index=True)
    out = out.sample(frac=1.0, random_state=42).reset_index(drop=True)
    out["src_id"] = out["src"].map(node_index)
    out["dst_id"] = out["dst"].map(node_index)
    return out

train_lp = make_labeled_edges(train_pos, train_neg)
val_lp = make_labeled_edges(val_pos, val_neg)

print("positive edges:", len(positive_pairs))
print("negative candidate pairs:", len(negative_pairs))
print("train link-pred rows:", len(train_lp))
print("validation link-pred rows:", len(val_lp))

display(train_lp)
display(val_lp)

In [ ]:
# JAX: embedding + logistic dot-product decoder
#
# score(u, v) = dot(emb_src[u], emb_dst[v]) + bias
# p(edge exists) = sigmoid(score)

def init_params(key, num_nodes: int, dim: int = 16):
    k1, k2 = jax.random.split(key)
    return {
        "emb": 0.05 * jax.random.normal(k1, (num_nodes, dim)),
        "bias": jnp.array(0.0),
    }

def edge_logits(params, src_ids, dst_ids):
    src_emb = params["emb"][src_ids]
    dst_emb = params["emb"][dst_ids]
    return jnp.sum(src_emb * dst_emb, axis=1) + params["bias"]

def bce_loss_from_logits(logits, labels):
    # stable binary cross entropy with logits
    return jnp.mean(jnp.maximum(logits, 0) - logits * labels + jnp.log1p(jnp.exp(-jnp.abs(logits))))

def loss_fn(params, src_ids, dst_ids, labels, l2=1e-4):
    logits = edge_logits(params, src_ids, dst_ids)
    bce = bce_loss_from_logits(logits, labels)
    reg = l2 * jnp.mean(params["emb"] ** 2)
    return bce + reg

@jax.jit
def sgd_step(params, src_ids, dst_ids, labels, lr):
    loss, grads = jax.value_and_grad(loss_fn)(params, src_ids, dst_ids, labels)
    new_params = jax.tree_util.tree_map(lambda p, g: p - lr * g, params, grads)
    return new_params, loss

num_nodes = len(node_index)
key = jax.random.PRNGKey(42)
params = init_params(key, num_nodes=num_nodes, dim=16)

train_src = jnp.array(train_lp["src_id"].to_numpy(), dtype=jnp.int32)
train_dst = jnp.array(train_lp["dst_id"].to_numpy(), dtype=jnp.int32)
train_y = jnp.array(train_lp["label"].to_numpy(), dtype=jnp.float32)

val_src = jnp.array(val_lp["src_id"].to_numpy(), dtype=jnp.int32)
val_dst = jnp.array(val_lp["dst_id"].to_numpy(), dtype=jnp.int32)
val_y = jnp.array(val_lp["label"].to_numpy(), dtype=jnp.float32)

loss_history = []
epochs = 1000
lr = 0.5

for epoch in range(epochs):
    params, loss = sgd_step(params, train_src, train_dst, train_y, lr)
    loss_history.append(float(loss))

print("final train loss:", loss_history[-1])

In [ ]:
# 推論・精度評価・可視化

def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))

def binary_auc(labels, scores):
    # sklearn なしで小さな validation set 用 AUC を計算する
    labels = np.asarray(labels)
    scores = np.asarray(scores)
    pos_scores = scores[labels == 1]
    neg_scores = scores[labels == 0]
    if len(pos_scores) == 0 or len(neg_scores) == 0:
        return np.nan
    comparisons = []
    for ps in pos_scores:
        for ns in neg_scores:
            if ps > ns:
                comparisons.append(1.0)
            elif ps == ns:
                comparisons.append(0.5)
            else:
                comparisons.append(0.0)
    return float(np.mean(comparisons))

val_logits = np.array(edge_logits(params, val_src, val_dst))
val_scores = sigmoid_np(val_logits)
val_pred = (val_scores >= 0.5).astype(int)
val_true = val_lp["label"].to_numpy()

accuracy = float((val_pred == val_true).mean())
auc = binary_auc(val_true, val_scores)

result_df = val_lp[["src", "dst", "label"]].copy()
result_df["score"] = val_scores
result_df["pred"] = val_pred
result_df["correct"] = result_df["label"] == result_df["pred"]

# 見やすい表示用
node_label_lookup = dict(zip(all_nodes["node_id"], all_nodes["raw_id"]))
result_df["src_raw"] = result_df["src"].map(node_label_lookup)
result_df["dst_raw"] = result_df["dst"].map(node_label_lookup)

display(
    result_df[
        ["src_raw", "dst_raw", "label", "score", "pred", "correct"]
    ].sort_values("score", ascending=False)
)

print(f"validation accuracy: {accuracy:.3f}")
print(f"validation AUC:      {auc:.3f}")

# 1. loss curve
plt.figure(figsize=(7, 4))
plt.plot(loss_history)
plt.xlabel("epoch")
plt.ylabel("binary cross entropy")
plt.title("Training loss")
plt.grid(True, alpha=0.3)
plt.show()

# 2. validation score visualization
plot_df = result_df.copy()
plot_df["edge"] = plot_df["src_raw"].str.replace("P", "P", regex=False) + " → " + plot_df["dst_raw"].str.replace("http://purl.obolibrary.org/obo/", "", regex=False)

plt.figure(figsize=(9, 4))
x = np.arange(len(plot_df))
plt.bar(x, plot_df["score"].to_numpy())
plt.axhline(0.5, linestyle="--")
plt.xticks(x, plot_df["edge"], rotation=45, ha="right")
plt.ylabel("predicted probability")
plt.title("Validation link prediction scores")
plt.tight_layout()
plt.show()

# 3. confusion matrix
tp = int(((val_true == 1) & (val_pred == 1)).sum())
tn = int(((val_true == 0) & (val_pred == 0)).sum())
fp = int(((val_true == 0) & (val_pred == 1)).sum())
fn = int(((val_true == 1) & (val_pred == 0)).sum())

cm = np.array([[tn, fp], [fn, tp]])

plt.figure(figsize=(4, 4))
plt.imshow(cm)
plt.xticks([0, 1], ["pred 0", "pred 1"])
plt.yticks([0, 1], ["true 0", "true 1"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")
plt.title("Validation confusion matrix")
plt.colorbar()
plt.tight_layout()
plt.show()

### 補足

このセル群は、かなり小さい toy data に対する transductive link prediction です。

実務的に拡張するなら、次の順で強くできます。

1. patient / phenotype の feature を追加する
2. `hasPhenotype` だけでなく `subClassOf` を message passing に使う
3. jraph / Flax で GraphSAGE / R-GCN 相当の encoder に置き換える
4. negative sampling を relation-aware にする
5. validation を patient-wise split / time-wise split にする

今回の notebook では、まず **ontology から作った graph を JAX の link prediction タスクに落とし、検証用データで推論スコアと精度を見る**ところまでを最小構成で追加しています。